# pgmq: a message queue as a Postgres extension

[pgmq](https://github.com/pgmq/pgmq) is a queue in the style of Amazon SQS, written as a Postgres
extension in plain SQL. A queue is a table, and every operation is a function call:
`pgmq.send`, `pgmq.read`, `pgmq.delete`, `pgmq.archive`. There is no client library and no worker
framework, so anything that can run SQL can produce or consume, from psql to a trigger.

The idea that carries everything is the **visibility timeout**. `read` does not lock a message or
mark it as owned by a worker; it pushes the message's `vt` timestamp into the future, and until
then nobody else sees it. A consumer that finishes deletes (or archives) the message. A consumer
that crashes does nothing, and when `vt` passes the message is simply there again.

`pgqueuer_playground.ipynb` covers the other design in this lab: pgqueuer marks a job `picked` and
its worker keeps a `heartbeat` column fresh while the job runs, and a stale heartbeat is what
frees the job. The last section puts the two side by side.

Everything here runs as plain SQL through psycopg, on queues named `nb_*`, which are dropped at
the end.

**Prerequisites**: pg18 is up (`just db pg18`). Its image builds pgmq 1.13 from source
(`docker/Dockerfile.pg18`); the pg16 image does not have it.

In [1]:
# Parameters. Papermill overrides these with `-p NAME value`.
SERVER = "pg18"  # a server in src/settings.py whose image has pgmq

In [2]:
import pathlib
import statistics
import sys
import threading
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

from dotenv import load_dotenv

ROOT = next(
    p
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "src" / "db.py").exists()
)
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")

import pandas as pd
import psycopg
from psycopg.types.json import Jsonb

from settings import settings

DSN = settings.sqlalchemy_url(SERVER).replace("postgresql+psycopg://", "postgresql://")
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 60)

# autocommit: every pgmq call is its own transaction, as it would be from a consumer process
conn = psycopg.connect(DSN, autocommit=True)


def q(sql: str, *args) -> pd.DataFrame:
    """Run one statement and show what came back."""
    cur = conn.execute(sql, args or None)
    if cur.description is None:
        return pd.DataFrame()
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def drop_nb_queues() -> None:
    for (name,) in conn.execute("SELECT queue_name FROM pgmq.list_queues() WHERE queue_name LIKE 'nb%'"):
        conn.execute("SELECT pgmq.drop_queue(%s)", (name,))


conn.execute("CREATE EXTENSION IF NOT EXISTS pgmq")
for (pattern, queue) in conn.execute("SELECT pattern, queue_name FROM pgmq.list_topic_bindings() WHERE queue_name LIKE 'nb%'").fetchall():
    conn.execute("SELECT pgmq.unbind_topic(%s, %s)", (pattern, queue))
drop_nb_queues()
q("SELECT extversion AS pgmq, current_setting('server_version') AS postgres FROM pg_extension WHERE extname = 'pgmq'")

,pgmq,postgres
0,1.13.0,18.6 (Debian 18.6-1.pgdg12+2)


## 1. A queue is two tables

`pgmq.create` makes a queue table `pgmq.q_<name>` and an archive table `pgmq.a_<name>`. A message
is a row: an id, the JSON body, optional headers, how many times it has been read, and `vt`, the
moment it becomes visible.

In [3]:
conn.execute("SELECT pgmq.create('nb_jobs')")
q("""
SELECT table_name, string_agg(column_name || ' ' || data_type, ', ' ORDER BY ordinal_position) AS columns
FROM information_schema.columns
WHERE table_schema = 'pgmq' AND table_name IN ('q_nb_jobs', 'a_nb_jobs')
GROUP BY table_name
""")

,table_name,columns
0,a_nb_jobs,"msg_id bigint, read_ct integer, enqueued_at timestamp wi..."
1,q_nb_jobs,"msg_id bigint, read_ct integer, enqueued_at timestamp wi..."


## 2. send and read

`send` inserts a row and returns its id. `read(queue, vt, qty)` takes up to `qty` visible
messages, oldest first, and moves their `vt` that many seconds ahead. It claims them with
`FOR UPDATE SKIP LOCKED` inside one `UPDATE`, so two consumers calling `read` at the same moment
never get the same row; the claim is committed as soon as `read` returns, and from then on only
`vt` keeps the message hidden.

In [4]:
ids = conn.execute(
    "SELECT * FROM pgmq.send_batch('nb_jobs', %s)",
    ([Jsonb({"page": p}) for p in ["orm/quickstart", "core/engines", "orm/session"]],),
).fetchall()
print("sent", [i for (i,) in ids])
q("SELECT msg_id, read_ct, vt, message FROM pgmq.read('nb_jobs', 30, 1)")

sent [1, 2, 3]


,msg_id,read_ct,vt,message
0,1,1,2026-10-17 13:02:42.443944+00:00,{'page': 'orm/quickstart'}


In [5]:
# the first message is now hidden for 30 s: the table still has it, with vt in the future
q("SELECT msg_id, read_ct, vt > now() AS hidden, message FROM pgmq.q_nb_jobs ORDER BY msg_id")

,msg_id,read_ct,hidden,message
0,1,1,True,{'page': 'orm/quickstart'}
1,2,0,False,{'page': 'core/engines'}
2,3,0,False,{'page': 'orm/session'}


In [6]:
# so a second read skips it and gets the other two
q("SELECT msg_id, read_ct, message FROM pgmq.read('nb_jobs', 30, 5)")

,msg_id,read_ct,message
0,2,1,{'page': 'core/engines'}
1,3,1,{'page': 'orm/session'}


## 3. The visibility timeout is the crash recovery

A consumer reads a message with a 2 second timeout and then dies without deleting it. Nothing has
to notice: no lock is held, no worker row goes stale. After 2 seconds `read` sees the message
again, with `read_ct` counting the attempt.

In [7]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.send('nb_jobs', %s)", (Jsonb({"page": "orm/relationships"}),))

crashed = q("SELECT msg_id, read_ct FROM pgmq.read('nb_jobs', 2, 1)")
print("consumer A read it, then crashed:", crashed.to_dict("records"))
print("right away, consumer B gets:", q("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 1)").to_dict("records"))
time.sleep(2.2)
q("SELECT msg_id, read_ct, message FROM pgmq.read('nb_jobs', 30, 1)")

consumer A read it, then crashed: [{'msg_id': 4, 'read_ct': 1}]
right away, consumer B gets: []


,msg_id,read_ct,message
0,4,2,{'page': 'orm/relationships'}


The same message, `read_ct` 2: delivery is **at least once**. A consumer that is merely slow gets
the same treatment as one that crashed, so the timeout has to be longer than the work, or the
consumer has to extend it (section 5).

## 4. Finishing a message: delete, archive or pop

- `delete` removes the row;
- `archive` moves it to `pgmq.a_<name>` with an `archived_at` time, a free audit log;
- `pop` reads and deletes in one statement: **at most once**, since a crash after `pop` loses
  the message.

In [8]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.send_batch('nb_jobs', %s)", ([Jsonb({"n": n}) for n in range(1, 5)],))
a, b = [r[0] for r in conn.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 2)")]
conn.execute("SELECT pgmq.delete('nb_jobs', %s::bigint)", (a,))
conn.execute("SELECT pgmq.archive('nb_jobs', %s::bigint)", (b,))
popped = q("SELECT msg_id, message FROM pgmq.pop('nb_jobs', 1)")
print("deleted", a, "archived", b, "popped", popped.msg_id.tolist())
print("left in the queue:", q("SELECT msg_id, message FROM pgmq.q_nb_jobs").to_dict("records"))
q("SELECT msg_id, read_ct, archived_at, message FROM pgmq.a_nb_jobs")

deleted 5 archived 6 popped [7]
left in the queue: [{'msg_id': 8, 'message': {'n': 4}}]


,msg_id,read_ct,archived_at,message
0,6,1,2026-10-17 13:02:14.723756+00:00,{'n': 2}


## 5. Long work: extending the timeout by hand

`set_vt(queue, msg_id, seconds)` moves `vt` to that many seconds from now. A consumer working on
something long calls it every so often, which is the heartbeat pgqueuer runs for you. Here the
work takes 3 seconds against a 2 second timeout, and a second consumer keeps trying to steal it.

In [9]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.send('nb_jobs', %s)", (Jsonb({"page": "orm/large"}),))


def thief(stop: threading.Event, got: list) -> None:
    with psycopg.connect(DSN, autocommit=True) as c:
        while not stop.is_set():
            got += [r[0] for r in c.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 1)")]
            time.sleep(0.1)


def work(extend: bool) -> list:
    conn.execute("SELECT pgmq.set_vt('nb_jobs', msg_id, 0) FROM pgmq.q_nb_jobs")  # visible again
    (msg_id,) = conn.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 2, 1)").fetchone()
    stop, stolen = threading.Event(), []
    t = threading.Thread(target=thief, args=(stop, stolen))
    t.start()
    for _ in range(6):  # 6 steps of 0.5 s
        time.sleep(0.5)
        if extend:
            conn.execute("SELECT pgmq.set_vt('nb_jobs', %s::bigint, 2)", (msg_id,))
    stop.set()
    t.join()
    return stolen


print("without set_vt, the other consumer also got:", work(extend=False))
print("with set_vt every 0.5 s, it got:", work(extend=True))

without set_vt, the other consumer also got: [9]


with set_vt every 0.5 s, it got: []


## 6. Retries, backoff and a dead-letter queue

pgmq has no retry policy; it has `read_ct` and `set_vt`, which are enough to write one. A failed
message is put back with a delay that grows with `read_ct`, and after three attempts it moves to
`nb_dead` in the same transaction that deletes it, so it is never in both queues or in neither.

In [10]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.create('nb_dead')")
pages = ["ok/1", "flaky/2", "ok/3", "broken/4"]
conn.execute("SELECT pgmq.send_batch('nb_jobs', %s)", ([Jsonb({"page": p}) for p in pages],))
MAX_ATTEMPTS = 3
log = []


def handle(page: str, attempt: int) -> None:
    if page.startswith("broken") or (page.startswith("flaky") and attempt < 2):
        raise RuntimeError(f"{page} failed")


t0 = time.perf_counter()
while conn.execute("SELECT count(*) FROM pgmq.q_nb_jobs").fetchone()[0]:
    for msg_id, read_ct, message in conn.execute("SELECT msg_id, read_ct, message FROM pgmq.read('nb_jobs', 30, 10)").fetchall():
        try:
            handle(message["page"], read_ct)
            conn.execute("SELECT pgmq.archive('nb_jobs', %s::bigint)", (msg_id,))
            log.append((round(time.perf_counter() - t0, 2), message["page"], read_ct, "done"))
        except RuntimeError:
            if read_ct >= MAX_ATTEMPTS:
                with conn.transaction():
                    conn.execute("SELECT pgmq.send('nb_dead', %s, %s)", (Jsonb(message), Jsonb({"attempts": read_ct})))
                    conn.execute("SELECT pgmq.delete('nb_jobs', %s::bigint)", (msg_id,))
                log.append((round(time.perf_counter() - t0, 2), message["page"], read_ct, "dead letter"))
            else:
                backoff = 0.25 * 2**read_ct  # 0.5 s, then 1 s
                conn.execute("SELECT pgmq.set_vt('nb_jobs', %s::bigint, %s::integer)", (msg_id, round(backoff)))
                log.append((round(time.perf_counter() - t0, 2), message["page"], read_ct, f"retry in {round(backoff)} s"))
    time.sleep(0.1)

display(pd.DataFrame(log, columns=["t", "page", "attempt", "outcome"]))
q("SELECT msg_id, message, headers FROM pgmq.q_nb_dead")

,t,page,attempt,outcome
0,0.00,ok/1,1,done
1,0.01,flaky/2,1,retry in 0 s
2,0.01,ok/3,1,done
3,0.01,broken/4,1,retry in 0 s
4,0.31,flaky/2,2,done
5,0.41,broken/4,2,retry in 1 s
6,33.04,broken/4,3,dead letter


,msg_id,message,headers
0,1,{'page': 'broken/4'},{'attempts': 3}


`set_vt` takes whole seconds, so the 0.5 s backoff rounds to 0 (an immediate retry) and the next
to 1 s; a `timestamptz` overload exists for finer control.

## 7. Several consumers on one queue

Four threads, each with its own connection, read batches of 10 from 400 messages and delete what
they finish. `FOR UPDATE SKIP LOCKED` inside `read` is what keeps them from claiming the same
rows; the check at the end counts every id exactly once.

In [11]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.send_batch('nb_jobs', %s)", ([Jsonb({"n": n}) for n in range(400)],))


def consumer(name: str) -> list[tuple[str, int]]:
    done = []
    with psycopg.connect(DSN, autocommit=True) as c:
        while batch := c.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 10)").fetchall():
            ids = [i for (i,) in batch]
            time.sleep(0.005)  # the work
            c.execute("SELECT pgmq.delete('nb_jobs', %s::bigint[])", (ids,))
            done += [(name, i) for i in ids]
    return done


t0 = time.perf_counter()
with ThreadPoolExecutor(4) as pool:
    results = [r for part in pool.map(consumer, ["c1", "c2", "c3", "c4"]) for r in part]
elapsed = time.perf_counter() - t0
seen = Counter(i for _, i in results)
print(f"{len(results)} deliveries of {len(seen)} distinct ids in {elapsed:.2f} s; any id twice: {any(v > 1 for v in seen.values())}")
pd.Series(Counter(name for name, _ in results)).rename("messages").to_frame().T

400 deliveries of 400 distinct ids in 0.12 s; any id twice: False


,c1,c2,c3,c4
messages,100,100,100,100


## 8. Waiting for messages: polling, read_with_poll, NOTIFY

An idle consumer has to find out that a message arrived. There are three ways:

- **poll** `read` from the client in a loop with a sleep: the latency is the sleep;
- **`read_with_poll(queue, vt, qty, max_poll_seconds, poll_interval_ms)`**: the same loop, run
  inside Postgres, so the client makes one call that returns as soon as something is there (or
  after `max_poll_seconds` with nothing);
- **`enable_notify_insert(queue, throttle_interval_ms)`**: a trigger sends `NOTIFY` on the channel
  `pgmq.q_<name>.INSERT` for every insert, at most once per throttle interval. The consumer
  `LISTEN`s and calls `read` when woken.

Each trial below waits on an empty queue, a message is sent after a short pause, and the latency is
from `send` returning to the consumer holding the message.

In [12]:
def trial(wait) -> float:
    conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
    got = {}
    ready = threading.Event()
    t = threading.Thread(target=lambda: got.update(t=wait(ready)))
    t.start()
    ready.wait()
    time.sleep(0.3)
    conn.execute("SELECT pgmq.send('nb_jobs', '{}'::jsonb)")
    sent = time.perf_counter()
    t.join()
    return (got["t"] - sent) * 1000


def client_poll(ready, sleep_s=0.2):
    with psycopg.connect(DSN, autocommit=True) as c:
        ready.set()
        while not c.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 1)").fetchall():
            time.sleep(sleep_s)
        return time.perf_counter()


def server_poll(ready):
    with psycopg.connect(DSN, autocommit=True) as c:
        ready.set()
        c.execute("SELECT msg_id FROM pgmq.read_with_poll('nb_jobs', 30, 1, 5, 100)").fetchall()
        return time.perf_counter()


def notified(ready):
    with psycopg.connect(DSN, autocommit=True) as c:
        c.execute('LISTEN "pgmq.q_nb_jobs.INSERT"')
        ready.set()
        for _ in c.notifies(timeout=5, stop_after=1):
            pass
        c.execute("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 1)").fetchall()
        return time.perf_counter()


conn.execute("SELECT pgmq.enable_notify_insert('nb_jobs', 0)")
rows = []
for label, fn in [("client poll, 200 ms sleep", client_poll), ("read_with_poll, 100 ms interval", server_poll), ("LISTEN / NOTIFY", notified)]:
    ms = [trial(fn) for _ in range(8)]
    rows.append({"wait": label, "median ms": round(statistics.median(ms), 1), "max ms": round(max(ms), 1)})
conn.execute("SELECT pgmq.disable_notify_insert('nb_jobs')")
pd.DataFrame(rows)

,wait,median ms,max ms
0,"client poll, 200 ms sleep",110.4,112.7
1,"read_with_poll, 100 ms interval",6.1,8.7
2,LISTEN / NOTIFY,4.7,5.4


A polling consumer finds the message at its next check, so the wait is anywhere up to the
interval (the worst case in the table), and which it is depends on where in the cycle the message
lands. NOTIFY wakes the consumer within milliseconds every time, the same mechanism pgqueuer uses
on `ch_pgqueuer`. The throttle (250 ms by default; 0 here) caps how often
a busy queue sends notifications, and a consumer should still `read` until the queue is empty
after each wake-up, since one notification can stand for many inserts.

## 9. Delayed messages

`send(queue, msg, delay)` sets `vt` that many seconds ahead at insert, so the message exists but
is invisible until then: the same column, used from the other end.

In [13]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("SELECT pgmq.send('nb_jobs', %s, 2)", (Jsonb({"remind": "rebuild index"}),))
print("now:", q("SELECT msg_id FROM pgmq.read('nb_jobs', 30, 1)").to_dict("records"))
time.sleep(2.1)
q("SELECT msg_id, enqueued_at, message FROM pgmq.read('nb_jobs', 30, 1)")

now: []


,msg_id,enqueued_at,message
0,438,2026-10-17 13:03:02.862791+00:00,{'remind': 'rebuild index'}


## 10. FIFO groups

Order across a whole queue is only best effort once several consumers are involved. pgmq 1.13 adds
SQS-style message groups: put a group key in the `x-pgmq-group` header and read with
`read_grouped`. A group's messages come out in order, and while one of them is in flight the rest
of that group stay hidden, so two consumers never work on the same group at once.
`read_grouped_rr` takes one message per group, round-robin, for fairness between groups.

In [14]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
for n, account in [(1, "acme"), (2, "acme"), (3, "globex"), (4, "acme"), (5, "globex"), (6, "initech")]:
    conn.execute("SELECT pgmq.send('nb_jobs', %s, %s)", (Jsonb({"n": n}), Jsonb({"x-pgmq-group": account})))

first = q("SELECT msg_id, message, headers->>'x-pgmq-group' AS grp FROM pgmq.read_grouped('nb_jobs', 30, 1)")
print("consumer A takes:", first.to_dict("records"))
print("consumer B, while acme's head is in flight:")
q("SELECT msg_id, message, headers->>'x-pgmq-group' AS grp FROM pgmq.read_grouped('nb_jobs', 30, 10)")

consumer A takes: [{'msg_id': 439, 'message': {'n': 1}, 'grp': 'acme'}]
consumer B, while acme's head is in flight:


,msg_id,message,grp
0,441,{'n': 3},globex
1,443,{'n': 5},globex
2,444,{'n': 6},initech


In [15]:
conn.execute("SELECT pgmq.set_vt('nb_jobs', msg_id, 0) FROM pgmq.q_nb_jobs")  # all visible again
print("round-robin, one per group:")
display(q("SELECT msg_id, message, headers->>'x-pgmq-group' AS grp FROM pgmq.read_grouped_rr('nb_jobs', 30, 3)"))
print("a plain read ignores groups altogether:")
q("SELECT msg_id, message, headers->>'x-pgmq-group' AS grp FROM pgmq.read('nb_jobs', 30, 10)")

round-robin, one per group:


,msg_id,message,grp
0,439,{'n': 1},acme
1,441,{'n': 3},globex
2,444,{'n': 6},initech


a plain read ignores groups altogether:


,msg_id,message,grp
0,440,{'n': 2},acme
1,442,{'n': 4},acme
2,443,{'n': 5},globex


The guarantee holds only if every consumer of the queue reads with a grouped function; one plain
`read` takes whatever is visible.

## 11. Topics: one send, several queues

`bind_topic(pattern, queue)` subscribes a queue to routing keys, with RabbitMQ-style wildcards:
`*` is exactly one dot-separated word, `#` any number. `send_topic(key, msg)` copies the message
into every bound queue and returns how many matched; `test_routing` shows the matches without
sending.

In [16]:
for name in ["nb_audit", "nb_eu", "nb_payments"]:
    conn.execute("SELECT pgmq.create(%s)", (name,))
for pattern, queue in [("orders.#", "nb_audit"), ("orders.eu.*", "nb_eu"), ("orders.*.paid", "nb_payments")]:
    conn.execute("SELECT pgmq.bind_topic(%s, %s)", (pattern, queue))

rows = []
for key in ["orders.eu.paid", "orders.us.paid", "orders.eu.refund.partial", "orders"]:
    matched = [r[1] for r in conn.execute("SELECT * FROM pgmq.test_routing(%s)", (key,))]
    n = conn.execute("SELECT pgmq.send_topic(%s, %s)", (key, Jsonb({"key": key}))).fetchone()[0]
    rows.append({"routing key": key, "delivered to": ", ".join(sorted(matched)) or "(none)", "copies": n})
pd.DataFrame(rows)

,routing key,delivered to,copies
0,orders.eu.paid,"nb_audit, nb_eu, nb_payments",3
1,orders.us.paid,"nb_audit, nb_payments",2
2,orders.eu.refund.partial,nb_audit,1
3,orders,(none),0


`orders.#` matches `orders.eu.paid` but not a bare `orders`: the pattern compiles to the regex
`^orders\..*$`, which wants at least one word after the dot.

## 12. Enqueue in the same transaction as the data

Because a queue is a table in the same database, a message can be sent in the transaction that
writes the row it refers to. If the transaction rolls back, so does the message: no job is ever
created for data that does not exist, and none is lost for data that does.

In [17]:
conn.execute("SELECT pgmq.purge_queue('nb_jobs')")
conn.execute("CREATE TABLE IF NOT EXISTS nb_documents (id serial PRIMARY KEY, title text)")

for title, fail in [("committed doc", False), ("rolled back doc", True)]:
    try:
        with conn.transaction():
            (doc_id,) = conn.execute("INSERT INTO nb_documents (title) VALUES (%s) RETURNING id", (title,)).fetchone()
            conn.execute("SELECT pgmq.send('nb_jobs', %s)", (Jsonb({"embed": doc_id, "title": title}),))
            if fail:
                raise RuntimeError("embedding service down, abort")
    except RuntimeError as e:
        print("rolled back:", e)

display(q("SELECT * FROM nb_documents"))
q("SELECT msg_id, message FROM pgmq.q_nb_jobs")

rolled back: embedding service down, abort


,id,title
0,1,committed doc


,msg_id,message
0,445,"{'embed': 1, 'title': 'committed doc'}"


## 13. Metrics

`metrics(queue)` and `metrics_all()` report what an SQS console would: queue length, the age of
the oldest and newest message, and the total ever sent.

In [18]:
q("SELECT queue_name, queue_length, newest_msg_age_sec, oldest_msg_age_sec, total_messages FROM pgmq.metrics_all() WHERE queue_name LIKE 'nb%' ORDER BY 1")

,queue_name,queue_length,newest_msg_age_sec,oldest_msg_age_sec,total_messages
0,nb_audit,3,0,0,3
1,nb_dead,1,20,20,1
2,nb_eu,1,0,0,1
3,nb_jobs,1,0,0,446
4,nb_payments,2,0,0,2


## 14. pgmq and pgqueuer side by side

Both claim work with `FOR UPDATE SKIP LOCKED` and neither holds a lock while the work runs; they
differ in how a claim expires and in where the logic lives.

| | pgmq | pgqueuer |
| --- | --- | --- |
| What it is | a Postgres extension, SQL functions | a Python library over plain tables |
| Claim | `read` pushes `vt` ahead | dequeue sets `status = 'picked'` |
| A claim expires | when `vt` passes, unless the consumer calls `set_vt` | when the worker's `heartbeat` is older than `heartbeat_timeout` (30 s), refreshed for you |
| Delivery | at least once (`pop`: at most once) | at least once |
| Consumers | any language with a Postgres driver | Python (asyncio) workers |
| Waking up | `read_with_poll`, or `enable_notify_insert` + `LISTEN` | `LISTEN ch_pgqueuer`, always on |
| Retries | by hand from `read_ct` and `set_vt` (section 6) | `RetryRequested`, retry executors, `on_failure="hold"` |
| Ordering | FIFO groups (`x-pgmq-group`) | priority, then id |
| Extras | archive table, topics with wildcards, delayed send, partitioned queues | dedupe keys, cross-worker concurrency limits, cron schedules, dashboard |
| Install | build the extension into the image (no PGDG package for pg18 yet) | `pgq install` creates its tables in any Postgres |

pgmq suits a queue shared by several languages or services, or one fed from SQL itself (a
trigger, a cron job in `pg_cron`), where an SQS-shaped API is the point. pgqueuer suits a Python
application that wants a worker framework, with retries, limits and schedules already written.

## Cleanup

In [19]:
for (pattern, queue) in conn.execute("SELECT pattern, queue_name FROM pgmq.list_topic_bindings() WHERE queue_name LIKE 'nb%'").fetchall():
    conn.execute("SELECT pgmq.unbind_topic(%s, %s)", (pattern, queue))
drop_nb_queues()
conn.execute("DROP TABLE IF EXISTS nb_documents")
conn.close()
print("clean")

clean
